# Supplemental ViT Inference

Runs batch inference for a supplemental ViT checkpoint against a matching RRUFF benchmark HDF5 and reports top-1/3/5 accuracy. No W&B login required.

## Environment

Use the shared train/eval environment:

```bash
conda activate paper-ai-diffraction-train-eval
pip install -e .
```

The following packages are required (already present in `environment-train-eval.yml`):
- `torch`, `h5py`, `numpy`, `tqdm`, `pandas`

## Checkpoints

Download supplemental ViT checkpoints from Zenodo and place them under:
```
external/checkpoints/
```
See `reproducibility/checkpoint_manifest.csv` for the full list.

## RRUFF benchmark HDF5

Large benchmark files reside on Stampede3 scratch. See `reproducibility/dataset_manifest.csv` for paths and the matching checkpoint ↔ benchmark pairings.

In [ ]:
import os
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'configs').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root from {cwd}')

MODELS_DIR = REPO_ROOT / 'external' / 'checkpoints'

# ---------------------------------------------------------------------------
# Per-checkpoint configs — verified from W&B (nist-berkeley-ai-diffraction/ai-diffraction).
#
# Shared architecture across ALL 11 supplemental checkpoints:
#   embed_dim=256  depth=12  num_heads=8  patch_size=25  mlp_ratio=4
#   use_mlp_head=False  num_classes=99
#
# Per-checkpoint variation: spec_length, use_rope, 2θ range, matching RRUFF HDF5.
# ---------------------------------------------------------------------------

CHECKPOINT_CONFIGS = {

    # ── Table S7: VT Training-Distribution Effect ───────────────────────────
    # 10M PyXtal, 5–90°, spec_length=8501; eval on RRUFF low-bkg full (8501 pts)

    'tite66yo': dict(  # VT-Balanced  (RRUFF-type label distribution)
        ckpt='xrd_model_tite66yo.pth',
        spec_length=8501, use_rope=False,
        two_theta_min=5.0, two_theta_max=90.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_low_bkg_full_intensity.hdf5',
    ),
    'k0it6fz2': dict(  # VT-ICSD  (ICSD label distribution)
        ckpt='xrd_model_k0it6fz2.pth',
        spec_length=8501, use_rope=False,
        two_theta_min=5.0, two_theta_max=90.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_low_bkg_full_intensity.hdf5',
    ),
    'l0yy9wv9': dict(  # VT-RRUFF  (RRUFF label distribution)
        ckpt='xrd_model_l0yy9wv9.pth',
        spec_length=8501, use_rope=False,
        two_theta_min=5.0, two_theta_max=90.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_low_bkg_full_intensity.hdf5',
    ),
    'cpv8k8ha': dict(  # VT-Augmented  (augmented RRUFF-type distribution)
        ckpt='xrd_model_cpv8k8ha.pth',
        spec_length=8501, use_rope=False,
        two_theta_min=5.0, two_theta_max=90.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_low_bkg_full_intensity.hdf5',
    ),

    # ── Table S3: VT Bias Cross-Evaluation ──────────────────────────────────
    # 10M reflection or PyXtal, 5–90°, spec_length=1000; eval on RRUFF full (1000 pts)

    'f3sdux88': dict(  # VT-Biased Reflection  (ICSD-biased label dist; crashed run)
        ckpt='xrd_model_f3sdux88.pth',
        spec_length=1000, use_rope=False,
        two_theta_min=5.0, two_theta_max=90.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_norm_uniform_sampling_5_90_deg_1000_intensity.hdf5',
    ),
    'kd1znx23': dict(  # VT-Biased PyXtal  (ICSD-biased label dist)
        ckpt='xrd_model_kd1znx23.pth',
        spec_length=1000, use_rope=False,
        two_theta_min=5.0, two_theta_max=90.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_norm_uniform_sampling_5_90_deg_1000_intensity.hdf5',
    ),
    'mth7zg2w': dict(  # VT-Balanced Reflection / VT-Full Reflection  (Tables S3/S5; crashed run)
        ckpt='xrd_model_mth7zg2w.pth',
        spec_length=1000, use_rope=False,
        two_theta_min=5.0, two_theta_max=90.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_norm_uniform_sampling_5_90_deg_1000_intensity.hdf5',
    ),
    '3zmiyil8': dict(  # VT-Balanced PyXtal / VT-Full PyXtal  (Tables S3/S5; use_rope=True)
        ckpt='xrd_model_3zmiyil8.pth',
        spec_length=1000, use_rope=True,
        two_theta_min=5.0, two_theta_max=90.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_norm_uniform_sampling_5_90_deg_1000_intensity.hdf5',
    ),

    # ── Tables S4/S5: VT Ablation — Data Generation Method and 2θ Range ────
    # Reduced 2θ range (10–80°); eval on matching reduced-range RRUFF benchmarks

    'smqmqi14': dict(  # VT-Reduced Reflection  (10–80°, spec_length=701; crashed run; also Fig S3)
        ckpt='xrd_model_smqmqi14.pth',
        spec_length=701, use_rope=False,
        two_theta_min=10.0, two_theta_max=80.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_norm_reduced_range_10_80_deg_701_intensity.hdf5',
    ),
    'bj1m83oz': dict(  # VT-Reduced PyXtal  (10–80°, spec_length=700)
        ckpt='xrd_model_bj1m83oz.pth',
        spec_length=700, use_rope=False,
        two_theta_min=10.0, two_theta_max=80.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_norm_reduced_range_10_80_deg_700_intensity.hdf5',
    ),

    # ── Fig S5: VT Attention Overlay ─────────────────────────────────────────
    # Use csv_to_db_with_bravais_vit_model.ipynb for the attention figure itself.
    # Listed here for completeness; top-1/3/5 inference is also valid.

    'pi7r8pah': dict(  # VT-Augmented attention model  (Table S7 / Fig S5)
        ckpt='xrd_model_pi7r8pah.pth',
        spec_length=8501, use_rope=False,
        two_theta_min=5.0, two_theta_max=90.0,
        rruff_h5='/scratch/10486/dchansew/Data/RRUFF_low_bkg_full_intensity.hdf5',
    ),
}

# ---------------------------------------------------------------------------
# Set the run ID to evaluate, then run all cells below.
# ---------------------------------------------------------------------------
RUN_ID = 'smqmqi14'

cfg        = CHECKPOINT_CONFIGS[RUN_ID]
ckpt_file  = cfg['ckpt']
RRUFF_H5   = cfg['rruff_h5']
SPEC_LENGTH     = cfg['spec_length']
USE_ROPE        = cfg['use_rope']
TWO_THETA_MIN   = cfg['two_theta_min']
TWO_THETA_MAX   = cfg['two_theta_max']
CHECKPOINT = str(MODELS_DIR / ckpt_file)

print(f'Run ID     : {RUN_ID}')
print(f'Checkpoint : {CHECKPOINT}')
print(f'Benchmark  : {RRUFF_H5}')
print(f'spec_length: {SPEC_LENGTH}  use_rope: {USE_ROPE}  2θ: {TWO_THETA_MIN}–{TWO_THETA_MAX}°')

In [ ]:
# Shared architecture for ALL 11 supplemental ViT checkpoints (verified from W&B).
# spec_length and use_rope come from the selected CHECKPOINT_CONFIGS entry above.
MODEL_CONFIG = dict(
    spec_length         = SPEC_LENGTH,
    num_output          = 99,
    patch_size          = 25,
    embed_dim           = 256,
    depth               = 12,
    num_heads           = 8,
    mlp_ratio           = 4.0,
    drop_ratio          = 0.0,
    use_rope            = USE_ROPE,
    use_mlp_head        = False,
    mlp_head_hidden_dim = 512,
)

NUM_CLASSES = 99
BATCH_SIZE  = 128
NUM_WORKERS = 4
START_COL   = 1
END_COL     = SPEC_LENGTH

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / 'src'))

import torch
import numpy as np
from collections import OrderedDict
from tqdm.auto import tqdm

from paper_ai_diffraction.core.model import VIT_model
from paper_ai_diffraction.core.dataset import get_dataloaders_test

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = VIT_model(**MODEL_CONFIG)

ckpt = torch.load(CHECKPOINT, map_location=device)
state_dict = ckpt.get('model', ckpt)
clean = OrderedDict((k.replace('module.', ''), v) for k, v in state_dict.items())
model.load_state_dict(clean, strict=False)
model = model.to(device)
model.eval()
print(f'Loaded: {ckpt_file}')

In [ ]:
_, _, test_loader, _, _ = get_dataloaders_test(
    h5_file       = RRUFF_H5,
    batch_size    = BATCH_SIZE,
    num_classes   = NUM_CLASSES,
    num_workers   = NUM_WORKERS,
    prefetch_factor = 2,
    start_col     = START_COL,
    end_col       = END_COL,
    label_mode    = 'categorical',
)
print(f'Test samples: {len(test_loader.dataset)}')

In [ ]:
top1_correct = top3_correct = top5_correct = total = 0

with torch.no_grad():
    for inputs, targets in tqdm(test_loader, desc='Evaluating'):
        inputs  = inputs.to(device)
        targets = targets.to(device)
        outputs = model(inputs)               # (B, 99)
        B = targets.size(0)
        total += B

        # top-k: targets in top-k predictions?
        _, topk = outputs.topk(5, dim=1)      # (B, 5)
        t = targets.view(-1, 1)               # (B, 1)
        top1_correct += topk[:, :1].eq(t).any(1).sum().item()
        top3_correct += topk[:, :3].eq(t).any(1).sum().item()
        top5_correct += topk[:, :5].eq(t).any(1).sum().item()

print(f'\nResults for {RUN_ID} on {Path(RRUFF_H5).name}')
print(f'  Total samples : {total}')
print(f'  Top-1         : {100 * top1_correct / total:.2f}%')
print(f'  Top-3         : {100 * top3_correct / total:.2f}%')
print(f'  Top-5         : {100 * top5_correct / total:.2f}%')